# Chapter 7: Command-Line Data Manipulation

## 1. Introduction

In a hospital or research environment, you'll often encounter data from various sources: legacy lab machines that output simple text files, server logs, or ad-hoc reports. Tools like `awk`, `sed`, and `cut` are the digital equivalent of a Swiss Army knife, allowing you to quickly parse, clean, and analyze this data directly in the command line without needing to write a full script or open a heavy application. This section will equip you with these three powerful utilities. You will learn to use `cut` to extract columns, `sed` to find and replace text, and `awk` to perform complex, logic-based filtering, turning raw text into structured, actionable insights.

---

## 2. Key Concepts and Definitions

Understanding these core tools and terms is crucial for effectively manipulating text data in a clinical or research context.

*   **`cut`**: A command-line utility that extracts sections (or "columns") from each line of a file based on a delimiter. Its medical context is in isolating specific data fields, like pulling only patient IDs and glucose values from a comprehensive lab report.
*   **`sed` (Stream Editor)**: A powerful utility for performing text transformations on an input stream (a file or input from a pipeline). It operates line-by-line and is commonly used for substitution (find and replace), such as standardizing drug names across multiple records.
*   **`awk`**: A versatile programming language designed for advanced text processing. It is field-aware and supports conditional logic, making it ideal for scanning patient vital signs and flagging only those that meet specific clinical criteria.
*   **Delimiter**: A character that separates distinct data fields. In a CSV (Comma-Separated Values) file, the delimiter is a comma. This is the key `cut` and `awk` use to understand the data's structure.
*   **Field**: A single piece of data in a record, separated from other fields by a delimiter. For example, in the line `PT-101,Smith,110`, "PT-101" is the first field.
*   **Pipe (`|`)**: A command-line feature that sends the standard output of one command to the standard input of another. This allows you to chain tools together to create powerful, multi-step data processing workflows.
*   **NSAID**: An acronym for Non-Steroidal Anti-Inflammatory Drug, a class of medications that includes Ibuprofen.
*   **Leukopenia**: A medical condition characterized by a low white blood cell (WBC) count, which can indicate a compromised immune system and increase the risk of infection.

---

## 3. Main Content

We will use the following example data files throughout our demonstrations. Note the standardized `PT-XXX` format for patient IDs.

**Example Data Files:**

**`patient_labs.csv`:**
```
PID,Name,Glucose (mg/dL),WBC (x10^9/L)
PT-101,Smith,110,7.5
PT-102,Jones,95,8.1
PT-103,Davis,140,3.8
```

**`prescriptions.txt`:**
```
Patient PT-101: Ibuprofen 200mg
Patient PT-102: Paracetamol 500mg
Patient PT-101: Take Ibuprofen 400mg; avoid other Ibuprofen products.
```

**`vitals.log`:**
```
2024-10-26 08:00 PT-102 HR 92
2024-10-26 08:00 PT-102 TEMP 37.1
2024-10-26 09:00 PT-103 HR 95
2024-10-26 09:00 PT-102 TEMP 38.5
```

### 3.1 Using `cut` for Column Extraction

Use `cut` to extract specific columns from delimited text. This is ideal for isolating data like patient IDs or specific lab values from tabular reports.

In [ ]:
%%bash
cut -d ',' -f1,3 patient_labs.csv

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




*   **`-d ','`**: Sets the delimiter to a comma. Quoting the delimiter is a robust practice that prevents the shell from misinterpreting special characters.
*   **`-f1,3`**: Selects the first and third fields (PID and Glucose).

**Output:**
```
PID,Glucose (mg/dL)
PT-101,110
PT-102,95
PT-103,140
```

### 3.2 Using `sed` for Search and Replace

Use `sed` (Stream Editor) to find and replace text. This is highly effective for standardizing terminology, such as annotating a drug name with its class.

In [ ]:
%%bash
sed 's/Ibuprofen/Ibuprofen (NSAID)/g' prescriptions.txt

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




*   **`s/find/replace/g`**: The substitution command. The `g` flag ensures all occurrences on each line are replaced.

> **Medical Background:** Standardizing drug names (e.g., adding "(NSAID)") is crucial for automated analysis. It allows a system to group all non-steroidal anti-inflammatory drugs together to check for potential contraindications, even if different brand names or descriptions are used in the raw text.

**Output:**
```
Patient PT-101: Ibuprofen (NSAID) 200mg
Patient PT-102: Paracetamol 500mg
Patient PT-101: Take Ibuprofen (NSAID) 400mg; avoid other Ibuprofen (NSAID) products.
```

### 3.3 Using `awk` for Advanced Filtering and Reporting

Use `awk` for field-based processing with conditional logic. It is powerful for scanning records to flag clinically significant events.

> **Important:** `awk` uses 1-based indexing for fields, meaning the first field is `$1`, the second is `$2`, and so on. This is different from many programming languages like Python or JavaScript, which use 0-based indexing. Keeping this distinction in mind will prevent common "off-by-one" errors.

In [ ]:
%%bash
# Log format: Date   Time   PID     Metric Value
#             $1     $2     $3      $4     $5
awk '$4 == "HR" && $5 > 90 {print "High HR Alert for PID:", $3, "Value:", $5}' vitals.log

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




*   **Pattern**: `$4 == "HR" && $5 > 90` finds lines where field 4 is "HR" and field 5 exceeds 90. While clinical tachycardia is >100 bpm, a threshold of 90 is often used for early warning alerts.
*   **Action**: `{print ...}` executes on matching lines to print a formatted alert.

**Output:**
```
High HR Alert for PID: PT-102 Value: 92
High HR Alert for PID: PT-103 Value: 95
```

### 3.4 Combining Tools with Pipes

Chaining tools with pipes (`|`) creates workflows. This example finds high glucose readings, reformats the ID, and then extracts the desired columns.

In [ ]:
%%bash
awk -F',' 'NR > 1 && $3 > 125 {print $0}' patient_labs.csv | sed 's/^/FLAG-/' | cut -d',' -f1,3

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




A single `awk` command is often more efficient.

In [ ]:
%%bash
awk -F',' -v OFS=',' 'NR > 1 && $3 > 125 {print "FLAG-"$1, $3}' patient_labs.csv

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




> **Pro Tip:** While pipelines are powerful, each `|` creates a new process, which adds overhead. For tasks involving filtering and reformatting, a single, well-crafted `awk` command is almost always more efficient than a long chain of `cut`, `grep`, and `sed` commands.

---

## 4. Practice Exercises

### Exercise 1: Extracting Patient Demographics

**Objective:** Isolate patient names and their white blood cell counts from a lab report.
**Time:** 2 minutes
**Medical Context:** A clinician might want a quick list of patients and their WBC counts to screen for potential immune disorders without needing other lab values.

From `patient_labs.csv`, write a command to extract only the `Name` (column 2) and `WBC` (column 4).

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
cut -d ',' -f2,4 patient_labs.csv
```
**Explanation:** This command uses `-d ','` to specify the comma delimiter and `-f2,4` to select the second and fourth fields from the `patient_labs.csv` file.
**Key Learning:** Using `cut` is the simplest way to extract entire columns of data from a delimited file.


</div>
</details>

### Exercise 2: Standardizing Drug Names

**Objective:** Replace a common drug name with its generic equivalent in a prescription file.
**Time:** 2 minutes
**Medical Context:** Standardizing drug names is critical for accurate medication reconciliation. "Paracetamol" and "Acetaminophen" are two names for the same common analgesic.

Using `prescriptions.txt`, write a command to replace 'Paracetamol' with 'Acetaminophen'.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
sed 's/Paracetamol/Acetaminophen/g' prescriptions.txt
```
**Explanation:** The `sed` command substitutes every occurrence of "Paracetamol" with "Acetaminophen".
**Key Learning:** `sed` is the go-to tool for simple, line-based search-and-replace tasks.


</div>
</details>

### Exercise 3: Filtering for High Temperature

**Objective:** Scan a log file for patient temperatures indicative of a fever and format the output as a CSV line.
**Time:** 5 minutes
**Medical Context:** Automated monitoring of vital signs is used to generate alerts for conditions like fever (pyrexia). A temperature over 38.0°C is a common threshold.

Using `vitals.log`, find lines for `TEMP` (field 4) where the value (field 5) is over 38.0. Print the output with fields separated by a single comma, like this: `HighTemp,PT-102,38.5`

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
awk '$4 == "TEMP" && $5 > 38.0 {print "HighTemp," $3 "," $5}' vitals.log
```
**Explanation:** The `awk` command checks if field 4 is "TEMP" and field 5 is greater than 38.0. For matching lines, it prints the string "HighTemp", the patient ID from field 3, and the temperature from field 5, all separated by commas.
**Key Learning:** `awk` excels at conditional filtering based on the values in specific fields and allows for custom output formatting.


</div>
</details>

> **Medical Background:** Leukopenia is a condition characterized by a low white blood cell (WBC) count, which can indicate a compromised immune system. A WBC count below 4.0 x10^9/L is a common clinical threshold used to flag patients for further review.

### Exercise 4: Identifying Patients with Potential Leukopenia

**Objective:** Create a command to identify and flag patients with low white blood cell counts.
**Time:** 5 minutes
**Medical Context:** Identifying patients with leukopenia from lab reports is crucial for timely clinical follow-up.

Write a single `awk` command that processes `patient_labs.csv` to:
1.  Filter for records where the WBC count (field 4) is less than 4.0 (ignoring the header row).
2.  Output only the PID (field 1) and the WBC count (field 4), separated by a comma, with the line prepended with `FLAG-LEUKOPENIA,`.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
awk -F',' -v OFS=',' 'NR > 1 && $4 < 4.0 {print "FLAG-LEUKOPENIA", $1, $4}' patient_labs.csv
```
**Explanation:**
*   **`-F','`**: Sets the input field separator to a comma to parse the CSV.
*   **`-v OFS=','`**: Sets the *output* field separator to a comma. This ensures the final output is correctly comma-separated.
*   **`NR > 1`**: Skips the first line (the header row). `NR` stands for "Number of Records."
*   **`&& $4 < 4.0`**: This is the core clinical filter for leukopenia.
*   **`{print "FLAG-LEUKOPENIA", $1, $4}`**: If the line matches, this action is executed. It prints the literal flag, the patient ID, and the WBC count.

**Key Learning:** A single, well-crafted `awk` command is often more efficient and readable than a long chain of piped commands for tasks involving filtering, formatting, and field manipulation.


</div>
</details>

---

## 5. Practical Applications

*   **Cohort Identification for Clinical Trials:** A researcher can use `awk` to rapidly scan a CSV file with millions of patient lab results. A command like `awk -F',' '$3 > 125 && $4 < 4.0 {print $1}' patient_data.csv` could instantly generate a list of Patient IDs for individuals with both high glucose and low WBC, identifying a potential cohort for a metabolic syndrome study.
*   **Genomic Data Formatting:** Variant Call Format (VCF) files in genomics are tab-delimited. Bioinformaticians frequently use `cut -f1,2,4,5` to extract specific columns (chromosome, position, reference/alternate alleles) to prepare data for downstream analysis. They might use `sed` to fix formatting inconsistencies in file headers before processing.
*   **Live Log File Alerting:** A data manager for a clinical system could use `tail -f vitals.log | awk '/TEMP/ && $5 > 38.5 {print "URGENT: Fever alert for PID " $3 | "mail -s 'Alert' admin@hospital.org"}'` to create a live alert system that immediately flags and emails dangerously high temperatures, enabling rapid clinical response.

---

## 6. Summary and Key Takeaways

In this section, we've explored how to perform fundamental text processing tasks from the command line using `cut`, `sed`, and `awk`. Mastering these tools is essential for any data-facing role in precision health.

*   **`cut`** is your tool for simple, column-based data extraction. It's fast and straightforward for when you just need specific fields.
*   **`sed`** is ideal for stream-based find-and-replace operations. Use it to standardize terminology or correct errors line-by-line.
*   **`awk`** is a complete processing language that shines at complex, field-aware logic. Use it when you need to apply conditional rules and generate custom reports.
*   **Efficiency Matters:** While you can chain tools with pipes, a single `awk` command is often more efficient for tasks that involve both filtering and reformatting.

> **Reflection Moment:** Think about a data-cleaning task you've faced, whether in a spreadsheet or a text file. Which of these three tools—`cut`, `sed`, or `awk`—would have been most helpful, and why? What specific operation would you have performed?

With these foundational data wrangling skills, you are now prepared to build more complex automations by creating your own shell scripts.

---


---

## 📝 Interactive Practice

Practice the concepts with these interactive exercises:

### Use `cut` to extract specific columns from delimited text. This is ideal for isolating data like patient IDs or specific lab values from tabular reports.

In [ ]:
%%bash
cut -d ',' -f1,3 patient_labs.csv

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### Use `sed` (Stream Editor) to find and replace text. This is highly effective for standardizing terminology, such as annotating a drug name with its class.

In [ ]:
%%bash
sed 's/Ibuprofen/Ibuprofen (NSAID)/g' prescriptions.txt

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### is `$2`, and so on. This is different from many programming languages like Python or JavaScript, which use 0-based indexing. Keeping this distinction in mind will prevent common "off-by-one" errors.

In [ ]:
%%bash
# Log format: Date   Time   PID     Metric Value
#             $1     $2     $3      $4     $5
awk '$4 == "HR" && $5 > 90 {print "High HR Alert for PID:", $3, "Value:", $5}' vitals.log

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### Chaining tools with pipes (`|`) creates workflows. This example finds high glucose readings, reformats the ID, and then extracts the desired columns.

In [ ]:
%%bash
awk -F',' 'NR > 1 && $3 > 125 {print $0}' patient_labs.csv | sed 's/^/FLAG-/' | cut -d',' -f1,3

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### A single `awk` command is often more efficient.

In [ ]:
%%bash
awk -F',' -v OFS=',' 'NR > 1 && $3 > 125 {print "FLAG-"$1, $3}' patient_labs.csv

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
cut -d ',' -f2,4 patient_labs.csv

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
sed 's/Paracetamol/Acetaminophen/g' prescriptions.txt

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
awk '$4 == "TEMP" && $5 > 38.0 {print "HighTemp," $3 "," $5}' vitals.log

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
awk -F',' -v OFS=',' 'NR > 1 && $4 < 4.0 {print "FLAG-LEUKOPENIA", $1, $4}' patient_labs.csv

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above


